# 02 — Random Forest

Este notebook define un clasificador Random Forest configurable mediante scikit-learn. Un **ensamble** combina varios modelos para producir una respuesta conjunta. Aquí cada modelo es un árbol de decisión que aprende con variaciones aleatorias de los datos y de los atributos disponibles. La combinación suele reducir la variación de un árbol individual, aunque su desempeño debe medirse en casos que no participaron del aprendizaje.

El alcance comprende la construcción del modelo, la transformación de atributos categóricos, una política explícita para datos faltantes, el entrenamiento y la adaptación de la predicción a tablas de pandas. Las celdas ejecutables solo importan herramientas y definen operaciones; no cargan datasets, ejecutan ejemplos ni calculan resultados experimentales.

### Cómo se construye el bosque

Para entrenar cada uno de los $B$ árboles, el **bootstrap** extrae $n$ casos con reemplazo de los $n$ casos de entrenamiento. Algunos aparecen varias veces y otros no aparecen. La probabilidad de que un caso quede fuera es $(1-1/n)^n$, aproximadamente $e^{-1}\simeq0{,}368$ cuando $n$ es grande. Si se desactiva bootstrap, cada árbol recibe el conjunto completo.

En cada nodo se considera un subconjunto aleatorio de los $M$ atributos disponibles. `max_features` controla su tamaño: por ejemplo, `sqrt` considera aproximadamente $\sqrt M$. La aleatoriedad busca que los árboles no repitan todos los mismos errores. La **semilla** fija la secuencia aleatoria para poder repetir un entrenamiento en el mismo entorno.

Los árboles del bosque usan divisiones binarias. Si $p_c$ es la proporción de la clase $c$ en un nodo, los criterios disponibles son:

$$\operatorname{Gini}(D)=1-\sum_c p_c^2,\qquad H(D)=-\sum_c p_c\log_2p_c.$$

Gini mide mezcla de clases y entropía mide incertidumbre. Para una división en izquierda y derecha se busca reducir la impureza:

$$\Delta I=I(D)-\frac{n_{izq}}n I(D_{izq})-\frac{n_{der}}n I(D_{der}).$$

`gini` utiliza Gini; `entropy` y `log_loss` utilizan el criterio de entropía para buscar divisiones. Estos árboles no seleccionan divisiones mediante Gain Ratio.

### Cómo produce una respuesta

Cada árbol $b$ devuelve en su hoja la proporción $P_b(c\mid x)$ de casos de cada clase. Esta implementación promedia esas probabilidades y elige la mayor:

$$\overline P(c\mid x)=\frac1B\sum_{b=1}^{B}P_b(c\mid x),\qquad \widehat y=\arg\max_c\overline P(c\mid x).$$

No es necesariamente igual a contar únicamente la clase ganadora de cada árbol cuando las hojas mezclan clases. Ante un empate se conserva el orden de clases que usa el estimador. Agregar árboles puede estabilizar el promedio, pero aumenta el costo y no garantiza mejorar cada conjunto de prueba. Estos promedios no deben interpretarse como probabilidades calibradas.

### Preparación de atributos y decisiones de diseño

Se construye un **pipeline**, una secuencia que aplica siempre la misma preparación antes del bosque. Las columnas numéricas conservan su valor. Las categóricas se convierten con **one-hot**: una columna indicadora por categoría aprendida. Una categoría nueva se representa con ceros en el grupo correspondiente. No se interpreta un código de categoría como una medida numérica: debe llegar como texto, booleano o `category`.

El codificador y cualquier imputador se ajustan únicamente con entrenamiento. No se escalan los números: las divisiones de los árboles comparan valores con umbrales. `max_features` se aplica sobre las columnas transformadas, de modo que un atributo con muchas categorías puede aportar varias columnas. Se fija un único proceso de cálculo (`n_jobs=1`) para hacer más comparables los tiempos.

Se ofrecen dos políticas explícitas para faltantes:

- `rechazar`: exige entradas completas tanto para entrenar como para predecir. Es la política inicial.
- `imputar`: reemplaza números ausentes por la mediana aprendida en entrenamiento y categorías ausentes por el texto reservado `__FALTANTE__`. Una columna numérica totalmente vacía se conserva con valor 0. El marcador categórico se reserva para ausentes y no debe aparecer como categoría real. La imputación es parte del pipeline y se vuelve a aprender al reentrenar.

La política explícita evita que el comportamiento dependa del soporte nativo de faltantes de una versión de la biblioteca. Los datos originales no se modifican. Se rechazan infinitos, complejos, tablas vacías, nombres de columnas repetidos y cambios de columnas u orden.

### Configuración y contrato de uso

| Parámetro en español | Parámetro del estimador | Efecto y valor inicial |
|---|---|---|
| `numero_arboles` | `n_estimators` | Cantidad de árboles: 100. |
| `maximo_atributos` | `max_features` | Atributos candidatos por nodo: `sqrt`. Admite también `log2`, `None`, entero positivo o proporción entre 0 y 1. |
| `profundidad_maxima` | `max_depth` | Límite de niveles de cada árbol: `None`, sin límite. |
| `criterio` | `criterion` | Medida de impureza: `gini`, `entropy` o `log_loss`; inicialmente `gini`. |
| `usar_bootstrap` | `bootstrap` | Muestreo con reemplazo: `True`. |
| `semilla` | `random_state` | Secuencia aleatoria del bosque: 42. |
| `tratamiento_faltantes` | Preparación del pipeline | `rechazar` o `imputar`; inicialmente `rechazar`. |

Cada árbol conserva los mínimos de dos casos para dividir y uno por hoja. `crear_bosque` construye un pipeline sin entrenarlo. `entrenar_bosque` recibe características y objetivos como DataFrames alineados, con una columna `objetivo` sin nulos, y devuelve el pipeline ajustado. `crear_predictor` asocia ese modelo a una función `predecir(caracteristicas)` que devuelve una única columna `prediccion`, conserva índices y orden y no vuelve a entrenar. `crear_modelo` es un nombre compatible para construir el mismo modelo.

[PENDIENTE OBLIGATORIO — HU-13 Y HU-14: COMPLETAR LA COMPARACIÓN DE HIPERPARÁMETROS DEL BOSQUE (HU-13) Y DEFINIR LA CONFIGURACIÓN O LAS CONFIGURACIONES FINALES A INTEGRAR EN LA INTERFAZ (HU-14), APOYÁNDOSE EN VALIDACIÓN Y RESERVANDO PRUEBA PARA EVALUAR. LOS VALORES INICIALES SON UNA REFERENCIA REPRODUCIBLE; NO SE EXIGE DEMOSTRAR UN ÓPTIMO GLOBAL.]


In [1]:
# Tablas, números y comprobaciones de tipos para las entradas del modelo.
from numbers import Integral, Real
import numpy as np
import pandas as pd
from pandas.api.types import is_bool_dtype, is_complex_dtype, is_numeric_dtype

# Componentes de la secuencia de preparación y del bosque permitido.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.utils.validation import check_is_fitted


## 1. Validación y normalización de entradas

La validación detecta errores de formato antes de llegar al estimador. Normalizar faltantes significa representar las distintas formas de ausencia como `np.nan`; todavía no se reemplaza ningún dato por un valor aprendido. Los tipos originales se usan para decidir qué columnas son numéricas y cuáles categóricas.


In [2]:
def validar_entrada_bosque(caracteristicas, tratamiento_faltantes="rechazar"):
    """Comprueba un DataFrame y su política de faltantes sin alterar sus valores."""
    # Verificar estructura y política antes de inspeccionar los datos.
    if tratamiento_faltantes not in ("rechazar", "imputar"):
        raise ValueError("tratamiento_faltantes debe ser rechazar o imputar.")
    if not isinstance(caracteristicas, pd.DataFrame):
        raise TypeError("Las características deben ser un pandas.DataFrame.")
    if caracteristicas.empty:
        raise ValueError("La tabla debe contener casos y atributos.")
    if not caracteristicas.columns.is_unique or caracteristicas.columns.isna().any():
        raise ValueError("Las columnas deben tener nombres únicos y no nulos.")
    if tratamiento_faltantes == "rechazar" and caracteristicas.isna().any().any():
        raise ValueError("La política rechazar no admite valores faltantes.")

    # Admitir solo números finitos y reservar el marcador usado por la imputación.
    for columna in caracteristicas:
        serie = caracteristicas[columna]
        if is_complex_dtype(serie.dtype):
            raise ValueError("No se admiten números complejos.")
        for valor in serie.dropna():
            if isinstance(valor, complex) or (isinstance(valor, Real) and not np.isfinite(valor)):
                raise ValueError("Los números conocidos deben ser reales y finitos.")
            if tratamiento_faltantes == "imputar" and valor == "__FALTANTE__":
                raise ValueError("__FALTANTE__ está reservado para representar ausentes.")


def normalizar_faltantes(caracteristicas):
    """Crea una tabla con np.nan en todas las posiciones ausentes."""
    # La copia como objetos permite representar ausentes de texto y de números.
    copia = caracteristicas.astype(object)
    return copia.where(caracteristicas.notna(), np.nan)


def validar_parametros_bosque(numero_arboles, maximo_atributos, profundidad_maxima,
                             criterio, usar_bootstrap, semilla):
    """Rechaza configuraciones inválidas con mensajes legibles antes de entrenar."""
    # Árboles, profundidad y semilla se configuran con enteros, no booleanos.
    if (not isinstance(numero_arboles, Integral) or isinstance(numero_arboles, bool)
            or numero_arboles < 1):
        raise ValueError("numero_arboles debe ser un entero positivo.")
    if profundidad_maxima is not None and (
            not isinstance(profundidad_maxima, Integral) or isinstance(profundidad_maxima, bool)
            or profundidad_maxima < 1):
        raise ValueError("profundidad_maxima debe ser None o un entero positivo.")
    if (not isinstance(semilla, Integral) or isinstance(semilla, bool)
            or not 0 <= semilla < 2**32):
        raise ValueError("semilla debe ser un entero entre 0 y 2**32 - 1.")

    # Cantidad de atributos: regla, cantidad entera o proporción.
    atributos_validos = maximo_atributos is None
    if isinstance(maximo_atributos, str):
        atributos_validos = maximo_atributos in ("sqrt", "log2")
    elif isinstance(maximo_atributos, Integral) and not isinstance(maximo_atributos, bool):
        atributos_validos = maximo_atributos >= 1
    elif isinstance(maximo_atributos, Real) and not isinstance(maximo_atributos, bool):
        atributos_validos = 0 < maximo_atributos <= 1
    if not atributos_validos:
        raise ValueError("maximo_atributos debe ser sqrt, log2, None, entero o proporción válida.")
    if criterio not in ("gini", "entropy", "log_loss"):
        raise ValueError("criterio debe ser gini, entropy o log_loss.")
    if not isinstance(usar_bootstrap, bool):
        raise ValueError("usar_bootstrap debe ser True o False.")


## 2. Construcción y entrenamiento del pipeline

El pipeline mantiene juntos el tratamiento de ausentes, la codificación y el bosque. Al llamar al entrenamiento se aprenden las medianas y categorías y después los árboles. Al predecir se reutiliza todo lo aprendido. Incluso si una columna queda totalmente vacía se conserva su lugar mediante `keep_empty_features=True`.


In [3]:
def crear_bosque(caracteristicas, numero_arboles=100, maximo_atributos="sqrt",
                 profundidad_maxima=None, criterio="gini", usar_bootstrap=True,
                 semilla=42, tratamiento_faltantes="rechazar"):
    """Construye, sin entrenar, un pipeline con la configuración solicitada."""
    # Validar la configuración y recordar los tipos originales de las columnas.
    validar_entrada_bosque(caracteristicas, tratamiento_faltantes)
    validar_parametros_bosque(numero_arboles, maximo_atributos, profundidad_maxima,
                             criterio, usar_bootstrap, semilla)
    columnas_numericas = [columna for columna in caracteristicas
                          if is_numeric_dtype(caracteristicas[columna].dtype)
                          and not is_bool_dtype(caracteristicas[columna].dtype)]
    columnas_categoricas = [columna for columna in caracteristicas
                            if columna not in columnas_numericas]

    # Preparar cada tipo de columna; la imputación se ajustará solo al entrenar.
    pasos_categoricos = []
    preparacion_numerica = "passthrough"
    if tratamiento_faltantes == "imputar":
        pasos_categoricos.append(("imputar", SimpleImputer(
            strategy="constant", fill_value="__FALTANTE__", keep_empty_features=True)))
        preparacion_numerica = SimpleImputer(strategy="median", keep_empty_features=True)
    pasos_categoricos.append(("codificar", OneHotEncoder(
        handle_unknown="ignore", sparse_output=False)))
    preparador = ColumnTransformer([
        ("categoricas", Pipeline(pasos_categoricos), columnas_categoricas),
        ("numericas", preparacion_numerica, columnas_numericas),
    ])

    # Unir la preparación y los árboles en una sola secuencia reutilizable.
    modelo = Pipeline([
        ("normalizar", FunctionTransformer(normalizar_faltantes, feature_names_out="one-to-one")),
        ("preprocesador", preparador),
        ("random_forest", RandomForestClassifier(
            n_estimators=numero_arboles, max_features=maximo_atributos,
            max_depth=profundidad_maxima, criterion=criterio,
            bootstrap=usar_bootstrap, random_state=semilla, n_jobs=1,
            min_samples_split=2, min_samples_leaf=1)),
    ])

    # Guardar el contrato original y los valores efectivos para registrar la corrida.
    modelo.columnas_entrada_ = caracteristicas.columns.copy()
    modelo.columnas_numericas_ = columnas_numericas
    modelo.configuracion_ = dict(
        numero_arboles=numero_arboles, maximo_atributos=maximo_atributos,
        profundidad_maxima=profundidad_maxima, criterio=criterio,
        usar_bootstrap=usar_bootstrap, semilla=semilla,
        tratamiento_faltantes=tratamiento_faltantes,
        minimo_muestras_division=2, minimo_muestras_hoja=1, procesos=1)
    return modelo


def entrenar_bosque(caracteristicas, objetivos, **parametros):
    """Entrena con DataFrames alineados y devuelve el pipeline aprendido."""
    # La respuesta correcta permanece separada de las características.
    if not isinstance(objetivos, pd.DataFrame):
        raise TypeError("Los objetivos deben ser un pandas.DataFrame.")
    if (not isinstance(caracteristicas, pd.DataFrame)
            or list(objetivos.columns) != ["objetivo"]
            or not caracteristicas.index.equals(objetivos.index)):
        raise ValueError("Se necesitan características y objetivo con el mismo índice y orden.")
    if objetivos.isna().any().any():
        raise ValueError("Las respuestas correctas no deben contener faltantes.")

    # El estimador recibe un vector, conservando afuera el DataFrame original.
    modelo = crear_bosque(caracteristicas, **parametros)
    modelo.fit(caracteristicas, objetivos["objetivo"])
    return modelo


## 3. Predicción con el formato acordado

La función devuelta por `crear_predictor` valida la consulta y entrega una tabla asociada a un modelo ya entrenado. La salida contiene una fila por caso y conserva índices duplicados. La validación usa los tipos recordados en entrenamiento: un atributo numérico no puede cambiar a texto en una consulta.


In [4]:
def crear_predictor(modelo):
    """Asocia un bosque entrenado a una función que devuelve un DataFrame."""
    # Comprobar que el bosque aprendió antes de aceptar consultas.
    check_is_fitted(modelo.named_steps["random_forest"])
    columnas = modelo.columnas_entrada_.copy()
    tratamiento = modelo.configuracion_["tratamiento_faltantes"]

    def predecir(caracteristicas):
        """Clasifica casos conservando su índice y orden, sin volver a entrenar."""
        # Mantener la misma estructura y el significado de los atributos numéricos.
        validar_entrada_bosque(caracteristicas, tratamiento)
        if not caracteristicas.columns.equals(columnas):
            raise ValueError("La consulta debe conservar columnas y orden de entrenamiento.")
        for columna in modelo.columnas_numericas_:
            if any(not isinstance(valor, Real) or isinstance(valor, bool)
                   for valor in caracteristicas[columna].dropna()):
                raise ValueError(f"El atributo {columna!r} debe conservar su tipo numérico.")

        # El pipeline aplica su preparación aprendida y entrega una etiqueta por caso.
        etiquetas = modelo.predict(caracteristicas)
        return pd.DataFrame({"prediccion": etiquetas}, index=caracteristicas.index.copy())

    return predecir


# Nombre compatible para construir exactamente el mismo pipeline.
crear_modelo = crear_bosque
